In [ ]:
import os, sys
import argparse

from IPython.display import Markdown, display

display(Markdown("README.md"))

In [ ]:
import peano4
import exahype2

dimension = 2
width  = [10,10]
offset = [0,0]
end_time = 3.0
plot_interval = 0.1
patch_size = 10
maximum_cell_size = 0.1
space_order = 2
time_order = 2
solvers = {
  "fv-rusanov-global-fixed":exahype2.solvers.fv.rusanov.GlobalFixedTimeStep,
  "fv-rusanov-global-fixed-enclave":exahype2.solvers.fv.rusanov.GlobalFixedTimeStepWithEnclaveTasking,
  "fv-rusanov-global-adaptive":exahype2.solvers.fv.rusanov.GlobalAdaptiveTimeStep,
  "fv-rusanov-global-adaptive-enclave":exahype2.solvers.fv.rusanov.GlobalAdaptiveTimeStepWithEnclaveTasking,
  #"fv-rusanov-local-enclave":exahype2.solvers.fv.rusanov.LocalTimeStepWithEnclaveTasking,
  "rkdg":exahype2.solvers.rkdg.rusanov.GlobalAdaptiveTimeStep,
}
solver="fv-rusanov-global-adaptive"
unknowns = {
  "p":1,
  "v":dimension,
}
parameters = {
  "rho":2.7,
  "c":6.0,
  "sigma":0.1149,
  "t0":0.7,
  "M0":1000.0,
}

In [ ]:
initial_conditions = """
  Q[0] = 0.0;
  Q[1] = 0.0;
  Q[2] = 0.0;
"""
boundary_conditions = """
  Qoutside[0] = 0.0;
  Qoutside[1] = 0.0;
  Qoutside[2] = 0.0;
"""
flux = """
  const double K_0 = rho * c * c;
  if (normal == 0) {
    F[0] = -K_0 * Q[1];
    F[1] = -1.0 / rho * Q[0];
    F[2] = 0.0;
  } else {
    F[0] = -K_0 * Q[2];
    F[1] = 0.0;
    F[2] = -1.0 / rho * Q[0];
  }
"""
eigenvalues = """
  return c;
"""
source_term = """
  tarch::la::Vector<Dimensions,double> pointSourceLocation = {5, 5};
  if (tarch::la::norm2(volumeCentre-pointSourceLocation) < 0.1) {
    const double f = M0 * std::exp(-((t - t0) * (t - t0)) / (2.0 * sigma * sigma));
    S[0] = f;
    S[1] = 0.0;
    S[2] = 0.0;
  }
"""

In [ ]:
my_solver = solvers[solver](
    name="AcousticWaveSolver",
    patch_size=patch_size,
    unknowns = sum(unknowns.values()),
    auxiliary_variables=0,
    min_volume_h=0.0,
    max_volume_h=maximum_cell_size,
    time_step_relaxation = 0.01,
    #pde_terms_without_state = True
    #rk_order             = args.time_order,
    #polynomials          = exahype2.solvers.GaussLegendreBasis(args.space_order),
    #face_projections     = exahype2.solvers.rkdg.FaceProjections.Solution,
)

my_solver.set_implementation(
  initial_conditions   = initial_conditions,
  boundary_conditions  = boundary_conditions,
  flux                 = flux,
  eigenvalues          = eigenvalues,
  source_term          = source_term,
)

for key, value in parameters.items():
  my_solver.add_solver_constants("static constexpr double {} = {};".format(key, value))

In [ ]:
project = exahype2.Project( ["applications","exahype2", "acousticwave"], "AcousticWaveApplication", ".", executable="acoustic-wave-application-" + str(args.dim) + "d-patch-size-" + str(args.patch_size) + "-cell-size-" + str(args.cell_size))
project.add_solver( my_solver )

project.set_load_balancing( "toolbox::loadbalancing::RecursiveSubdivision", "new ::exahype2::LoadBalancingConfiguration()" )

project.set_global_simulation_parameters(
  dimensions            = dimension,
  size                  = width,
  offset                = offset,
  min_end_time          = end_time,
  max_end_time          = end_time,
  first_plot_time_stamp = 0.0,
  time_in_between_plots = plot_interval,
  periodic_BC           = [False, False, False]
)

In [ ]:
build_mode = peano4.output.CompileMode.Release
project.set_Peano4_installation("../../../", build_mode )

peano4_project = project.generate_Peano4_project(verbose=True)
peano4_project.generate(throw_away_data_after_generation=True)
peano4_project.build(make_clean_first=True)


In [ ]:
peano4_project.run( pipefile="log.txt", rebuild_if_required=False )